[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brilliantbeaver/alexpose/blob/main/gait/penny/gavd3/04_pretrain_sjepa_on_normal.ipynb)

# 04. Pretrain S-JEPA on normal gait

Train a compact, paper-aligned latent predictor on normal sequences only, with collapse checks and a reproducible checkpoint record.

**Research use only.** This tutorial does not diagnose a person or validate a clinical device.

**Run it:** locally, use `uv sync` then `uv run jupyter lab` from this folder. In Colab, use the badge and run the setup cell. Restart the kernel after changing the root `.env` file.

**Keep the walk visible:** notebook 01 opens the source video, and notebook 02 shows frame, bbox, and skeleton alignment. Revisit those views whenever a latent or classifier result looks surprising.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/brilliantbeaver/alexpose.git"

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "numpy", "pandas", "scipy", "scikit-learn", "matplotlib",
        "seaborn", "torch", "tqdm", "python-dotenv", "yt-dlp[default]",
        "opencv-python-headless", "mediapipe", "joblib", "pyarrow",
    ])
    clone_dir = Path("/content/alexpose")
    if not (clone_dir / ".git").exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)])
    os.chdir(clone_dir)


def find_project_root(start=None):
    env_root = os.getenv("ALEXPOSE_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / ".git").exists() and (candidate / "data" / "gavd").exists():
            return candidate
        print(f"Ignoring invalid ALEXPOSE_ROOT: {candidate}")
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists() and (candidate / "data" / "gavd").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()

try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env", override=False)
except Exception:
    pass

MODE = os.getenv("GAVD3_MODE", "smoke").strip().lower()
if MODE not in {"smoke", "real"}:
    raise ValueError("GAVD3_MODE must be smoke or real")
if MODE == "smoke":
    print(
        "SMOKE MODE: hand-authored motions test code paths only. "
        "They have no pathophysiological or clinical validity."
    )

PREFERRED_ROOT = Path(
    os.getenv(
        "GAVD4_ROOT",
        "/Users/pmui/vaults/worldmodels/gait/skeleton-jepa/gavd4",
    )
).expanduser()

requested_data = os.getenv("GAVD4_DATA_DIR") or os.getenv("GAVD_DATA_GAVD_DIR")
if requested_data and Path(requested_data).expanduser().exists():
    DATA_GAVD_DIR = Path(requested_data).expanduser()
elif requested_data:
    print(f"Ignoring missing GAVD CSV path: {Path(requested_data).expanduser()}")
    if (PREFERRED_ROOT / "data-gavd").exists():
        DATA_GAVD_DIR = PREFERRED_ROOT / "data-gavd"
    else:
        DATA_GAVD_DIR = PROJECT_ROOT / "data" / "gavd"
elif (PREFERRED_ROOT / "data-gavd").exists():
    DATA_GAVD_DIR = PREFERRED_ROOT / "data-gavd"
else:
    DATA_GAVD_DIR = PROJECT_ROOT / "data" / "gavd"

requested_youtube = os.getenv("GAVD4_YOUTUBE_DIR") or os.getenv("GAVD_YOUTUBE_DIR")
if requested_youtube:
    YOUTUBE_DIR = Path(requested_youtube).expanduser()
elif PREFERRED_ROOT.exists():
    YOUTUBE_DIR = PREFERRED_ROOT / "youtube"
else:
    YOUTUBE_DIR = PROJECT_ROOT / "gait" / "penny" / "gavd3" / "work" / "youtube"

TUTORIAL_DIR = PROJECT_ROOT / "gait" / "penny" / "gavd3"
CACHE_DIR = Path(
    os.getenv("GAVD3_CACHE_DIR", TUTORIAL_DIR / "work" / "cache")
).expanduser()
ARTIFACT_ROOT = Path(
    os.getenv("GAVD3_ARTIFACT_DIR", TUTORIAL_DIR / "work" / "artifacts")
).expanduser()
ARTIFACT_DIR = ARTIFACT_ROOT / MODE
POSE_DIR = ARTIFACT_DIR / "poses"

for folder in [CACHE_DIR, ARTIFACT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", str(CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_DIR / "xdg-cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
from IPython.display import SVG, display


def show_tutorial_svg(filename):
    '''Render a repository SVG reliably in local Jupyter and Colab.'''
    path = TUTORIAL_DIR / "images" / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Missing tutorial figure {path}. Clone the full alexpose repository."
        )
    display(SVG(filename=str(path)))

print(f"mode: {MODE}")
print(f"project: {PROJECT_ROOT}")
print(f"GAVD CSVs: {DATA_GAVD_DIR}")
print(f"YouTube cache: {YOUTUBE_DIR}")
print(f"artifacts: {ARTIFACT_DIR}")


In [ ]:
show_tutorial_svg("06_training_step.svg")


In [ ]:
BLAZEPOSE_33 = [
    "NOSE", "LEFT_EYE_INNER", "LEFT_EYE", "LEFT_EYE_OUTER",
    "RIGHT_EYE_INNER", "RIGHT_EYE", "RIGHT_EYE_OUTER", "LEFT_EAR",
    "RIGHT_EAR", "MOUTH_LEFT", "MOUTH_RIGHT", "LEFT_SHOULDER",
    "RIGHT_SHOULDER", "LEFT_ELBOW", "RIGHT_ELBOW", "LEFT_WRIST",
    "RIGHT_WRIST", "LEFT_PINKY", "RIGHT_PINKY", "LEFT_INDEX",
    "RIGHT_INDEX", "LEFT_THUMB", "RIGHT_THUMB", "LEFT_HIP",
    "RIGHT_HIP", "LEFT_KNEE", "RIGHT_KNEE", "LEFT_ANKLE",
    "RIGHT_ANKLE", "LEFT_HEEL", "RIGHT_HEEL", "LEFT_FOOT_INDEX",
    "RIGHT_FOOT_INDEX",
]
MASK_KEYPOINTS = [11, 12, 23, 24, 25, 26, 27, 28, 31, 32]
assert [BLAZEPOSE_33[i] for i in MASK_KEYPOINTS] == [
    "LEFT_SHOULDER", "RIGHT_SHOULDER", "LEFT_HIP", "RIGHT_HIP",
    "LEFT_KNEE", "RIGHT_KNEE", "LEFT_ANKLE", "RIGHT_ANKLE",
    "LEFT_FOOT_INDEX", "RIGHT_FOOT_INDEX",
]


In [ ]:
CONDITIONS = ["normal", "parkinsons", "stroke", "cerebralpalsy", "myopathic"]


In [ ]:
def synthetic_gait_sequence(condition="normal", frames=64, seed=0):
    '''Create a code-path fixture, not a physiological disease simulation.'''
    rng = np.random.default_rng(seed)
    phase = np.linspace(0.0, 4.0 * np.pi, frames, endpoint=False)
    seq = np.zeros((frames, 33, 4), dtype=np.float32)
    seq[..., 3] = 1.0
    base = {
        11: (0.42, 0.28), 12: (0.58, 0.28),
        23: (0.45, 0.52), 24: (0.55, 0.52),
        25: (0.44, 0.70), 26: (0.56, 0.70),
        27: (0.43, 0.89), 28: (0.57, 0.89),
        29: (0.42, 0.92), 30: (0.58, 0.92),
        31: (0.39, 0.94), 32: (0.61, 0.94),
    }
    for joint, (x, y) in base.items():
        seq[:, joint, 0] = x
        seq[:, joint, 1] = y
    amplitude = 0.045
    lift = 0.025
    if condition == "parkinsons":
        amplitude *= 0.45
        lift *= 0.45
    if condition == "myopathic":
        seq[:, [11, 12], 0] += 0.03 * np.sin(phase)[:, None]
        seq[:, [23, 24], 0] += 0.018 * np.sin(phase)[:, None]
    for joint, knee, foot, offset in [(27, 25, 31, 0.0), (28, 26, 32, np.pi)]:
        wave = np.sin(phase + offset)
        if condition == "stroke" and joint == 27:
            wave = 0.35 * wave
        if condition == "cerebralpalsy":
            seq[:, knee, 1] -= 0.045
            seq[:, joint, 1] -= 0.02
        seq[:, joint, 0] += amplitude * wave
        seq[:, knee, 0] += 0.4 * amplitude * wave
        seq[:, foot, 0] += amplitude * wave
        seq[:, joint, 1] -= lift * np.maximum(wave, 0.0)
        seq[:, foot, 1] -= 0.7 * lift * np.maximum(wave, 0.0)
    seq[..., :3] += rng.normal(0.0, 0.0025, seq[..., :3].shape)
    return seq


def synthetic_corpus(conditions=None, n_per_condition=10, frames=64, seed=42):
    if conditions is None:
        conditions = [
            "normal", "parkinsons", "stroke", "cerebralpalsy", "myopathic"
        ]
    records = []
    counter = 0
    for condition in conditions:
        for sample in range(n_per_condition):
            records.append({
                "condition": condition,
                "sequence_id": f"smoke_{condition}_{sample:03d}",
                "video_id": f"smoke_video_{condition}_{sample // 2:02d}",
                "sequence": synthetic_gait_sequence(
                    condition=condition,
                    frames=frames,
                    seed=seed + counter,
                ),
            })
            counter += 1
    return records


In [ ]:
def interpolate_low_visibility(sequence, threshold=0.45, max_gap=4):
    '''Fill only short internal gaps and preserve the original validity mask.

    Long gaps and sequence ends are never extrapolated. Their coordinates remain
    missing until center_and_scale converts them to an explicit zero sentinel.
    They can never become S-JEPA prediction targets.
    '''
    sequence = np.asarray(sequence, dtype=np.float32).copy()
    if sequence.ndim != 3 or sequence.shape[1:] != (33, 4):
        raise ValueError(f"Expected [T, 33, 4], received {sequence.shape}")
    visibility = np.nan_to_num(sequence[..., 3], nan=0.0)
    finite = np.isfinite(sequence[..., :3]).all(axis=-1)
    valid = (visibility >= threshold) & finite
    filled = valid.copy()
    for joint in range(33):
        observed = np.flatnonzero(valid[:, joint])
        for left, right in zip(observed[:-1], observed[1:]):
            gap = int(right - left - 1)
            if not 0 < gap <= max_gap:
                continue
            fraction = (
                np.arange(1, gap + 1, dtype=np.float32) / (gap + 1)
            )[:, None]
            sequence[left + 1:right, joint, :3] = (
                sequence[left, joint, :3][None, :] * (1.0 - fraction)
                + sequence[right, joint, :3][None, :] * fraction
            )
            filled[left + 1:right, joint] = True
        sequence[~filled[:, joint], joint, :3] = np.nan
    sequence[..., 3] = visibility
    return sequence, valid


def center_and_scale(sequence, eps=1e-6):
    sequence = np.asarray(sequence, dtype=np.float32).copy()
    xyz = sequence[..., :3]
    left_hip, right_hip = xyz[:, 23], xyz[:, 24]
    left_ok = np.isfinite(left_hip).all(axis=1)
    right_ok = np.isfinite(right_hip).all(axis=1)
    pelvis = np.full((len(xyz), 3), np.nan, dtype=np.float32)
    pelvis[left_ok & right_ok] = 0.5 * (
        left_hip[left_ok & right_ok] + right_hip[left_ok & right_ok]
    )
    pelvis[left_ok & ~right_ok] = left_hip[left_ok & ~right_ok]
    pelvis[right_ok & ~left_ok] = right_hip[right_ok & ~left_ok]
    pelvis_ok = np.isfinite(pelvis).all(axis=1)
    fallback = np.median(pelvis[pelvis_ok], axis=0) if pelvis_ok.any() else np.zeros(3)
    pelvis[~np.isfinite(pelvis).all(axis=1)] = fallback
    xyz = xyz - pelvis[:, None, :]
    shoulder_width = np.linalg.norm(xyz[:, 11, :2] - xyz[:, 12, :2], axis=-1)
    hip_width = np.linalg.norm(xyz[:, 23, :2] - xyz[:, 24, :2], axis=-1)
    body_scale = np.nanmedian(np.maximum(shoulder_width, hip_width))
    if not np.isfinite(body_scale) or body_scale < eps:
        body_scale = 1.0
    sequence[..., :3] = np.nan_to_num(
        xyz / body_scale, nan=0.0, posinf=0.0, neginf=0.0
    )
    return np.nan_to_num(sequence, nan=0.0, posinf=0.0, neginf=0.0)


def temporal_resize(array, frames):
    array = np.asarray(array)
    if len(array) == frames:
        return array.copy()
    if len(array) < 2:
        return np.repeat(array, frames, axis=0)
    old_t = np.linspace(0.0, 1.0, len(array))
    new_t = np.linspace(0.0, 1.0, frames)
    flat = array.reshape(len(array), -1)
    resized = np.stack(
        [np.interp(new_t, old_t, flat[:, i]) for i in range(flat.shape[1])],
        axis=1,
    )
    return resized.reshape(frames, *array.shape[1:]).astype(array.dtype)


def prepare_sequence(
    sequence,
    frames=64,
    visibility_threshold=0.45,
    max_gap=4,
):
    cleaned, valid = interpolate_low_visibility(
        sequence, visibility_threshold, max_gap=max_gap
    )
    cleaned = center_and_scale(cleaned)
    cleaned = temporal_resize(cleaned, frames)
    valid = temporal_resize(valid.astype(np.float32), frames) >= 0.5
    return cleaned[..., :3].astype(np.float32), valid.astype(bool)


In [ ]:
def uniform_neurologic_mask(valid_patch, mask_fraction=0.60, seed=None):
    """Sample eligible joint-time tokens uniformly, without motion scores.

    valid_patch has shape [B, S, V]. True means that a patch can be a target.
    The returned mask has the same shape. True means hidden from the view encoder.
    """
    valid_patch = np.asarray(valid_patch, dtype=bool)
    if valid_patch.ndim != 3 or valid_patch.shape[2] != 33:
        raise ValueError(f"Expected [B, S, 33], received {valid_patch.shape}")
    if not 0.0 < mask_fraction < 1.0:
        raise ValueError("mask_fraction must be between 0 and 1")
    rng = np.random.default_rng(seed)
    eligible_joint = np.zeros(33, dtype=bool)
    eligible_joint[MASK_KEYPOINTS] = True
    eligible = valid_patch & eligible_joint[None, None, :]
    counts = eligible.reshape(len(eligible), -1).sum(axis=1)
    if np.any(counts < 2):
        raise ValueError("Each sample needs at least two valid eligible tokens")
    n_mask = max(1, int(np.floor(counts.min() * mask_fraction)))
    n_mask = min(n_mask, int(counts.min()) - 1)
    mask = np.zeros_like(eligible)
    for batch_index in range(len(mask)):
        candidates = np.flatnonzero(eligible[batch_index].reshape(-1))
        chosen = rng.choice(candidates, size=n_mask, replace=False)
        mask[batch_index].reshape(-1)[chosen] = True
    forbidden = sorted(set(range(33)).difference(MASK_KEYPOINTS))
    assert not mask[:, :, forbidden].any()
    assert mask.reshape(len(mask), -1).any(axis=1).all()
    assert (~mask).reshape(len(mask), -1).any(axis=1).all()
    return mask


def mask_audit(mask, valid_patch):
    mask = np.asarray(mask, dtype=bool)
    valid_patch = np.asarray(valid_patch, dtype=bool)
    eligible_joint = np.zeros(33, dtype=bool)
    eligible_joint[MASK_KEYPOINTS] = True
    eligible = valid_patch & eligible_joint[None, None, :]
    masked_counts = mask.reshape(len(mask), -1).sum(axis=1)
    eligible_counts = eligible.reshape(len(mask), -1).sum(axis=1)
    per_sample_ratio = masked_counts / eligible_counts
    touched = np.flatnonzero(mask.any(axis=(0, 1))).tolist()
    return {
        "masked_keypoints": touched,
        "masked_names": [BLAZEPOSE_33[i] for i in touched],
        "global_fraction": float(mask.mean()),
        "eligible_mask_fraction_min": float(per_sample_ratio.min()),
        "eligible_mask_fraction_mean": float(per_sample_ratio.mean()),
        "eligible_mask_fraction_max": float(per_sample_ratio.max()),
        "forbidden_count": int(mask[:, :, sorted(set(range(33)) - set(MASK_KEYPOINTS))].sum()),
    }


In [ ]:
import copy
import math
import torch
from torch import nn


class SkeletonPatchEncoder(nn.Module):
    def __init__(
        self,
        frames=64,
        joints=33,
        coordinate_dim=3,
        segment_length=4,
        embed_dim=64,
        depth=2,
        heads=4,
        dropout=0.0,
    ):
        super().__init__()
        if frames % segment_length:
            raise ValueError("frames must be divisible by segment_length")
        self.frames = frames
        self.joints = joints
        self.coordinate_dim = coordinate_dim
        self.segment_length = segment_length
        self.segments = frames // segment_length
        self.embed_dim = embed_dim
        self.patch_embed = nn.Linear(segment_length * coordinate_dim, embed_dim)
        self.time_pos = nn.Parameter(torch.randn(self.segments, embed_dim) * 0.02)
        self.joint_pos = nn.Parameter(torch.randn(joints, embed_dim) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.blocks = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)

    def patchify(self, x):
        batch, frames, joints, channels = x.shape
        expected = (self.frames, self.joints, self.coordinate_dim)
        if (frames, joints, channels) != expected:
            raise ValueError(f"Expected [B, {expected}], received {x.shape}")
        patches = x.reshape(
            batch, self.segments, self.segment_length, joints, channels
        )
        patches = patches.permute(0, 1, 3, 2, 4).contiguous()
        return patches.flatten(3)

    def positioned_tokens(self, x):
        tokens = self.patch_embed(self.patchify(x))
        return (
            tokens
            + self.time_pos[None, :, None, :]
            + self.joint_pos[None, None, :, :]
        )

    def forward(self, x, keep_mask=None):
        tokens = self.positioned_tokens(x)
        batch = len(tokens)
        flat = tokens.reshape(batch, self.segments * self.joints, self.embed_dim)
        if keep_mask is not None:
            keep_mask = keep_mask.reshape(batch, -1)
            kept_per_sample = keep_mask.sum(dim=1)
            if not torch.equal(kept_per_sample, kept_per_sample[:1].expand_as(kept_per_sample)):
                raise ValueError("Each sample must keep the same number of tokens")
            flat = flat[keep_mask].reshape(batch, int(kept_per_sample[0]), self.embed_dim)
        return self.norm(self.blocks(flat))


class SkeletonPredictor(nn.Module):
    def __init__(
        self,
        segments,
        joints,
        encoder_dim=64,
        predictor_dim=64,
        depth=2,
        heads=4,
        dropout=0.0,
    ):
        super().__init__()
        self.segments = segments
        self.joints = joints
        self.encoder_to_predictor = nn.Linear(encoder_dim, predictor_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, predictor_dim))
        nn.init.normal_(self.mask_token, std=0.02)
        self.time_pos = nn.Parameter(torch.randn(segments, predictor_dim) * 0.02)
        self.joint_pos = nn.Parameter(torch.randn(joints, predictor_dim) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=predictor_dim,
            nhead=heads,
            dim_feedforward=predictor_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.blocks = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(predictor_dim)
        self.output = nn.Linear(predictor_dim, encoder_dim)

    def forward(self, visible_features, target_mask):
        batch = len(visible_features)
        target_mask = target_mask.reshape(batch, self.segments * self.joints)
        visible_mask = ~target_mask
        visible = self.encoder_to_predictor(visible_features)
        full = self.mask_token.expand(
            batch, self.segments * self.joints, -1
        ).clone()
        full[visible_mask] = visible.reshape(-1, visible.shape[-1])
        positions = (
            self.time_pos[:, None, :] + self.joint_pos[None, :, :]
        ).reshape(1, self.segments * self.joints, -1)
        full = full + positions
        predicted = self.output(self.norm(self.blocks(full)))
        return predicted[target_mask].reshape(batch, -1, predicted.shape[-1])


class SJEPAGait(nn.Module):
    def __init__(
        self,
        frames=64,
        joints=33,
        coordinate_dim=3,
        segment_length=4,
        embed_dim=64,
        encoder_depth=2,
        predictor_depth=2,
        heads=4,
    ):
        super().__init__()
        self.view_encoder = SkeletonPatchEncoder(
            frames, joints, coordinate_dim, segment_length,
            embed_dim, encoder_depth, heads,
        )
        self.target_encoder = copy.deepcopy(self.view_encoder)
        for parameter in self.target_encoder.parameters():
            parameter.requires_grad_(False)
        self.predictor = SkeletonPredictor(
            self.view_encoder.segments,
            joints,
            embed_dim,
            embed_dim,
            predictor_depth,
            heads,
        )
        self.register_buffer("target_center", torch.zeros(embed_dim))

    def forward(self, view, target, target_mask):
        visible_features = self.view_encoder(view, keep_mask=~target_mask)
        predicted = self.predictor(visible_features, target_mask)
        with torch.no_grad():
            all_targets = self.target_encoder(target)
            flat_mask = target_mask.reshape(len(target), -1)
            selected = all_targets[flat_mask].reshape(
                len(target), -1, all_targets.shape[-1]
            )
        return predicted, selected

    @torch.no_grad()
    def update_target(self, momentum):
        for target_parameter, view_parameter in zip(
            self.target_encoder.parameters(), self.view_encoder.parameters()
        ):
            target_parameter.mul_(momentum).add_(
                view_parameter, alpha=1.0 - momentum
            )

    @torch.no_grad()
    def update_center(self, targets, beta=0.9):
        batch_center = targets.mean(dim=(0, 1))
        self.target_center.mul_(beta).add_(batch_center, alpha=1.0 - beta)


def sjepa_cross_entropy(
    predicted,
    targets,
    center,
    predictor_temperature=0.10,
    target_temperature=0.06,
):
    target_prob = torch.softmax(
        (targets - center[None, None, :]) / target_temperature,
        dim=-1,
    ).detach()
    prediction_log_prob = torch.log_softmax(
        predicted / predictor_temperature,
        dim=-1,
    )
    return -(target_prob * prediction_log_prob).sum(dim=-1).mean()


def cosine_ema(step, total_steps, start=0.996, end=1.0):
    progress = min(max(step / max(total_steps - 1, 1), 0.0), 1.0)
    return end - (end - start) * (math.cos(math.pi * progress) + 1.0) / 2.0


LEFT_RIGHT_PAIRS = [
    (1, 4), (2, 5), (3, 6), (7, 8), (9, 10), (11, 12),
    (13, 14), (15, 16), (17, 18), (19, 20), (21, 22),
    (23, 24), (25, 26), (27, 28), (29, 30), (31, 32),
]


def geometric_view(
    x,
    max_degrees=8.0,
    translate=0.03,
    flip_probability=0.0,
):
    """Apply one sequence-wide transform per sample.

    Rotation is around the relative vertical y axis, so x and z are mixed.
    Flip defaults to off because laterality can matter for stroke. If enabled,
    coordinates are reflected and every left-right landmark pair is swapped.
    """
    view = x.clone()
    present = view.abs().sum(dim=-1) > 1e-8
    batch = len(view)
    angles = (
        torch.rand(batch, device=x.device) * 2.0 - 1.0
    ) * math.radians(max_degrees)
    cosine, sine = torch.cos(angles), torch.sin(angles)
    original_x = view[..., 0].clone()
    original_z = view[..., 2].clone()
    rotated_x = cosine[:, None, None] * original_x + sine[:, None, None] * original_z
    rotated_z = -sine[:, None, None] * original_x + cosine[:, None, None] * original_z
    view[..., 0] = rotated_x
    view[..., 2] = rotated_z
    offsets = (torch.rand(batch, 1, 1, 2, device=x.device) * 2.0 - 1.0) * translate
    view[..., :2] += offsets
    if flip_probability > 0:
        flip = torch.rand(batch, device=x.device) < flip_probability
        for batch_index in torch.where(flip)[0].tolist():
            view[batch_index, ..., 0] *= -1.0
            original = view[batch_index].clone()
            original_present = present[batch_index].clone()
            for left, right in LEFT_RIGHT_PAIRS:
                view[batch_index, :, left] = original[:, right]
                view[batch_index, :, right] = original[:, left]
                present[batch_index, :, left] = original_present[:, right]
                present[batch_index, :, right] = original_present[:, left]
    view = view.masked_fill(~present[..., None], 0.0)
    return view


In [ ]:
def pose_records_from_cache(pose_dir=POSE_DIR, conditions=CONDITIONS):
    records = []
    for condition in conditions:
        folder = Path(pose_dir) / condition
        for path in sorted(folder.glob("*.npz")):
            data = np.load(path, allow_pickle=False)
            required = {
                "sequence", "sequence_id", "video_id", "condition",
                "frame_numbers", "crop_bounds", "fps", "source_csv",
                "source_video", "pose_model", "pose_model_sha256",
                "extraction_version",
            }
            missing = required.difference(data.files)
            if missing:
                raise ValueError(
                    f"Stale pose cache {path} is missing {sorted(missing)}. "
                    "Re-extract it with notebook 02."
                )
            sequence = data["sequence"].astype(np.float32)
            if sequence.ndim != 3 or sequence.shape[1:] != (33, 4):
                raise ValueError(f"Bad pose shape in {path}: {sequence.shape}")
            stored_condition = str(data["condition"].item())
            if stored_condition != condition:
                raise ValueError(
                    f"Pose condition {stored_condition} does not match folder {condition}"
                )
            if len(data["frame_numbers"]) != len(sequence):
                raise ValueError(f"Frame and pose lengths differ in {path}")
            records.append({
                "condition": condition,
                "sequence_id": str(data["sequence_id"].item()),
                "video_id": str(data["video_id"].item()),
                "source_video": str(data["source_video"].item()),
                "fps": float(data["fps"].item()),
                "extraction_version": str(data["extraction_version"].item()),
                "pose_model_sha256": str(data["pose_model_sha256"].item()),
                "sequence": sequence,
                "path": str(path),
            })
    return records


def load_records_for_mode(conditions=CONDITIONS, smoke_per_condition=10, frames=64):
    if MODE == "smoke":
        records = synthetic_corpus(
            conditions=conditions,
            n_per_condition=smoke_per_condition,
            frames=frames,
        )
        print(f"Explicit smoke corpus: {len(records)} synthetic sequences")
        return records
    records = pose_records_from_cache(conditions=conditions)
    counts = pd.Series([r["condition"] for r in records]).value_counts()
    missing = [condition for condition in conditions if counts.get(condition, 0) == 0]
    if missing:
        raise FileNotFoundError(
            f"Real mode requires cached pose sequences for {missing}. "
            "Run notebook 02 first."
        )
    print(f"Real pose corpus: {len(records)} sequences")
    return records


## Tutorial scale and paper scale

|Setting|Tutorial default|ECCV paper|
|---|---:|---:|
|Frames|32 smoke, 64 real|120|
|Temporal segment|4|4|
|Encoder width|32 smoke, 96 real|256|
|Encoder depth|1 smoke, 4 real|8|
|Predictor depth|1 smoke, 2 real|5|
|Attention heads|4|8|
|Epochs|2 smoke, 20 real|1200|
|Global mask ratio|At most 10/33 by design|0.90|

The smaller defaults teach and debug the full computation graph on a laptop. They are not expected to reproduce NTU benchmark accuracy.


## Prepare only normal sequences

Visibility controls a maximum four-frame internal interpolation and target eligibility. Ends and longer gaps are never extrapolated. They become a zero sentinel after centering and can never become targets. Each clip is centered on the mid-hip, scaled by shoulder or hip width, temporally resized, and reduced to x, y, relative z.

Temporal resizing can weaken absolute cadence information. Keep FPS and duration as separate metadata if you add hybrid classifier features later.


In [ ]:
FRAMES = int(os.getenv("SJEPA_FRAMES", "32" if MODE == "smoke" else "64"))
SEGMENT_LENGTH = 4
if FRAMES % SEGMENT_LENGTH:
    raise ValueError("SJEPA_FRAMES must be divisible by 4")

normal_records = load_records_for_mode(
    conditions=["normal"],
    smoke_per_condition=8,
    frames=FRAMES,
)
assert {record["condition"] for record in normal_records} == {"normal"}
expected_normal = int(os.getenv("GAVD_EXPECTED_NORMAL_SEQUENCES", "12"))
if MODE == "real" and expected_normal and len(normal_records) != expected_normal:
    raise ValueError(
        f"Expected {expected_normal} normal pose files, found {len(normal_records)}. "
        "Run notebook 02 with GAVD_MAX_SEQUENCES=0."
    )
prepared = [
    prepare_sequence(record["sequence"], frames=FRAMES)
    for record in normal_records
]
normal_xyz = np.stack([item[0] for item in prepared])
normal_valid = np.stack([item[1] for item in prepared])
sequence_ids = [record["sequence_id"] for record in normal_records]
video_ids = [record["video_id"] for record in normal_records]
min_coverage = float(os.getenv("GAVD_MIN_NEURO_COVERAGE", "0.50"))
coverage_report = pd.DataFrame({
    "sequence_id": sequence_ids,
    "video_id": video_ids,
    "neurologic_observed_fraction": normal_valid[:, :, MASK_KEYPOINTS].mean(axis=(1, 2)),
})
display(coverage_report)
coverage_report.to_csv(
    ARTIFACT_DIR / "normal_pose_coverage.csv", index=False
)
below = coverage_report[
    coverage_report["neurologic_observed_fraction"] < min_coverage
]
if not below.empty:
    raise ValueError(
        f"{len(below)} normal sequences fall below the neurologic "
        f"coverage threshold {min_coverage:.2f}. Review extraction first."
    )

print("normal tensor:", normal_xyz.shape)
print("normal sequences:", len(sequence_ids))
print("normal source videos:", len(set(video_ids)))
if MODE == "real" and len(set(video_ids)) == 1:
    print(
        "Warning: all normal sequences share one source video. "
        "This pretraining run is transductive with respect to exp5 normal clips."
    )


## Configure the compact model

The target encoder starts as an exact copy of the view encoder and has requires_grad=False. The optimizer sees only the view encoder and predictor.

Real mode has two explicit profiles. `recommended` is the default substantive run for this very small corpus. It uses more optimizer updates and a slightly faster starting EMA than the large-scale S-JEPA recipe. `quick` checks the complete real-data code path but is not a representation result. Select it with `SJEPA_RUN_PROFILE=quick`. The exact choices are saved in the checkpoint fingerprint.


In [ ]:
if MODE == "smoke":
    RUN_PROFILE = "smoke"
    EMBED_DIM, ENCODER_DEPTH, PREDICTOR_DEPTH, HEADS = 32, 1, 1, 4
    EPOCHS = int(os.getenv("SJEPA_EPOCHS", "2"))
    BATCH_SIZE = 4
    EMA_START = 0.996
else:
    RUN_PROFILE = os.getenv(
        "SJEPA_RUN_PROFILE", "recommended"
    ).strip().lower()
    if RUN_PROFILE not in {"quick", "recommended"}:
        raise ValueError(
            "SJEPA_RUN_PROFILE must be quick or recommended"
        )
    EMBED_DIM, ENCODER_DEPTH, PREDICTOR_DEPTH, HEADS = 96, 4, 2, 4
    default_epochs = "20" if RUN_PROFILE == "quick" else "300"
    default_ema = "0.996" if RUN_PROFILE == "quick" else "0.999"
    EPOCHS = int(os.getenv("SJEPA_EPOCHS", default_epochs))
    BATCH_SIZE = int(os.getenv("SJEPA_BATCH_SIZE", "4"))
    EMA_START = float(os.getenv("SJEPA_EMA_START", default_ema))

config = {
    "frames": FRAMES,
    "joints": 33,
    "coordinate_dim": 3,
    "segment_length": SEGMENT_LENGTH,
    "embed_dim": EMBED_DIM,
    "encoder_depth": ENCODER_DEPTH,
    "predictor_depth": PREDICTOR_DEPTH,
    "heads": HEADS,
}
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
torch.manual_seed(42)
np.random.seed(42)
model = SJEPAGait(**config).to(device)
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    trainable,
    lr=3e-4 if MODE == "smoke" else 1e-3,
    betas=(0.9, 0.95),
    weight_decay=0.05,
)
print("device:", device)
print("training profile:", RUN_PROFILE)
if MODE == "real" and RUN_PROFILE == "quick":
    print("QUICK PROFILE: validate the pipeline only; do not report its score.")
print("trainable parameters:", sum(p.numel() for p in trainable))
print(
    "target trainable parameters:",
    sum(p.numel() for p in model.target_encoder.parameters() if p.requires_grad),
)
assert not any(p.requires_grad for p in model.target_encoder.parameters())


## Train and monitor representation health

Loss alone can look healthy during collapse. Each epoch also records:

- mean standard deviation across pooled latent dimensions
- mean off-diagonal cosine similarity between sequences
- target and predictor entropy
- realized global and valid-eligible mask fractions


In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F

dataset = TensorDataset(
    torch.tensor(normal_xyz, dtype=torch.float32),
    torch.tensor(normal_valid, dtype=torch.bool),
)
generator = torch.Generator().manual_seed(42)
loader = DataLoader(
    dataset,
    batch_size=min(BATCH_SIZE, len(dataset)),
    shuffle=True,
    generator=generator,
    drop_last=False,
)
total_steps = EPOCHS * len(loader)
warmup_steps = max(1, min(len(loader), total_steps // 10))
minimum_initial_teacher_retention = EMA_START ** total_steps
print("optimizer updates:", total_steps)
print(
    "minimum initial-teacher retention under the EMA schedule:",
    f"{minimum_initial_teacher_retention:.3f}",
)
print(
    "The actual retention is higher because momentum rises toward 1. "
    "This makes update count, not epoch count alone, the useful scale."
)


def learning_rate_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(total_steps - warmup_steps - 1, 1)
    return 0.5 + 0.5 * (1.0 + math.cos(math.pi * progress)) / 2.0


scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=learning_rate_factor
)


@torch.no_grad()
def collapse_diagnostics(model, arrays, batch_size=8):
    model.target_encoder.eval()
    pooled = []
    for start in range(0, len(arrays), batch_size):
        batch = torch.tensor(
            arrays[start:start + batch_size],
            dtype=torch.float32,
            device=device,
        )
        tokens = model.target_encoder(batch)
        pooled.append(tokens.mean(dim=1).cpu())
    pooled = torch.cat(pooled)
    feature_std = pooled.std(dim=0, unbiased=False).mean().item()
    normalized = F.normalize(pooled, dim=1)
    similarities = normalized @ normalized.T
    if len(pooled) > 1:
        off_diagonal = similarities[~torch.eye(
            len(pooled), dtype=torch.bool
        )].mean().item()
    else:
        off_diagonal = float("nan")
    return feature_std, off_diagonal


history = []
global_step = 0
target_before = next(model.target_encoder.parameters()).detach().clone()
for epoch in range(EPOCHS):
    model.train()
    batch_rows = []
    for coordinates, valid in loader:
        coordinates = coordinates.to(device)
        valid = valid.to(device)
        segments = FRAMES // SEGMENT_LENGTH
        valid_patch = (
            valid.reshape(
                len(valid), segments, SEGMENT_LENGTH, 33
            )
            .all(dim=2)
            .cpu()
            .numpy()
        )
        mask_np = uniform_neurologic_mask(
            valid_patch,
            mask_fraction=0.60,
            seed=42 + global_step,
        )
        mask_stats = mask_audit(mask_np, valid_patch)
        target_mask = torch.tensor(mask_np, device=device)
        view = geometric_view(
            coordinates,
            max_degrees=8.0,
            translate=0.03,
            flip_probability=0.0,
        )
        prediction, target = model(view, coordinates, target_mask)
        loss = sjepa_cross_entropy(
            prediction,
            target,
            model.target_center,
        )
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite loss at step {global_step}")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        assert all(
            parameter.grad is None
            for parameter in model.target_encoder.parameters()
        )
        torch.nn.utils.clip_grad_norm_(trainable, max_norm=1.0)
        optimizer.step()
        scheduler.step()
        momentum = cosine_ema(
            global_step, total_steps, start=EMA_START, end=1.0
        )
        model.update_target(momentum)
        model.update_center(target, beta=0.9)

        with torch.no_grad():
            centered = target - model.target_center[None, None, :]
            target_prob = torch.softmax(centered / 0.06, dim=-1)
            predicted_prob = torch.softmax(prediction / 0.10, dim=-1)
            target_entropy = (
                -(target_prob * target_prob.clamp_min(1e-8).log())
                .sum(dim=-1)
                .mean()
                .item()
            )
            predictor_entropy = (
                -(predicted_prob * predicted_prob.clamp_min(1e-8).log())
                .sum(dim=-1)
                .mean()
                .item()
            )
        batch_rows.append({
            "loss": float(loss.detach().cpu()),
            "target_entropy": target_entropy,
            "predictor_entropy": predictor_entropy,
            "global_mask_fraction": mask_stats["global_fraction"],
            "eligible_mask_fraction": mask_stats["eligible_mask_fraction_mean"],
            "ema_momentum": momentum,
        })
        global_step += 1

    feature_std, mean_cosine = collapse_diagnostics(model, normal_xyz)
    summary = pd.DataFrame(batch_rows).mean(numeric_only=True).to_dict()
    summary.update({
        "epoch": epoch + 1,
        "feature_std": feature_std,
        "mean_pair_cosine": mean_cosine,
        "learning_rate": optimizer.param_groups[0]["lr"],
    })
    history.append(summary)
    print(
        f"epoch {epoch + 1:03d}  "
        f"loss {summary['loss']:.4f}  "
        f"std {feature_std:.4f}  "
        f"cos {mean_cosine:.4f}"
    )

history_df = pd.DataFrame(history)
target_after = next(model.target_encoder.parameters()).detach()
ema_change = float((target_after - target_before).abs().mean().cpu())
assert ema_change > 0.0
assert np.isfinite(history_df.select_dtypes("number")).all().all()
print("mean absolute EMA target change:", ema_change)
display(history_df)


## Save a checkpoint with an audit trail

The fingerprint binds the checkpoint to the preprocessed tensor content, mode, IDs, model configuration, preprocessing rule, mask rule, and training settings. A downstream notebook refuses a checkpoint created in another mode or with another target set.


In [ ]:
import hashlib
import json

content_hasher = hashlib.sha256()
validity_hasher = hashlib.sha256()
for sequence_id, array, validity in sorted(
    zip(sequence_ids, normal_xyz, normal_valid),
    key=lambda item: item[0],
):
    content_hasher.update(sequence_id.encode("utf-8"))
    content_hasher.update(np.ascontiguousarray(array).tobytes())
    validity_hasher.update(sequence_id.encode("utf-8"))
    validity_hasher.update(
        np.ascontiguousarray(validity, dtype=np.bool_).tobytes()
    )
pose_provenance = sorted(
    [
        {
            "sequence_id": record["sequence_id"],
            "extraction_version": record.get(
                "extraction_version", "synthetic_fixture"
            ),
            "pose_model_sha256": record.get(
                "pose_model_sha256", "not_applicable"
            ),
        }
        for record in normal_records
    ],
    key=lambda item: item["sequence_id"],
)
fingerprint_payload = {
    "mode": MODE,
    "sequence_ids": sorted(sequence_ids),
    "video_ids": sorted(video_ids),
    "preprocessed_content_sha256": content_hasher.hexdigest(),
    "validity_mask_sha256": validity_hasher.hexdigest(),
    "pose_provenance": pose_provenance,
    "model_config": config,
    "mask_keypoints": MASK_KEYPOINTS,
    "mask_fraction": 0.60,
    "visibility_threshold": 0.45,
    "maximum_interpolation_gap": 4,
    "minimum_neurologic_coverage": min_coverage,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "run_profile": RUN_PROFILE,
    "seed": 42,
    "ema_start": EMA_START,
}
dataset_fingerprint = hashlib.sha256(
    json.dumps(fingerprint_payload, sort_keys=True).encode("utf-8")
).hexdigest()
checkpoint_path = ARTIFACT_DIR / "sjepa_normal.pt"
torch.save(
    {
        "model_state": model.state_dict(),
        "config": config,
        "mode": MODE,
        "mask_keypoints": MASK_KEYPOINTS,
        "dataset_fingerprint": dataset_fingerprint,
        "sequence_ids": sequence_ids,
        "video_ids": video_ids,
        "paper_aligned_core_components": [
            "temporal joint patches",
            "visible-token view encoder",
            "full-input EMA target encoder",
            "Transformer predictor",
            "target centering and sharpening",
            "latent cross-entropy",
        ],
        "fingerprint_payload": fingerprint_payload,
        "deliberate_adaptation": "uniform targets within neurologic keypoints only",
    },
    checkpoint_path,
)
history_df.to_csv(ARTIFACT_DIR / "training_history.csv", index=False)
print("checkpoint:", checkpoint_path)
print("fingerprint:", dataset_fingerprint)


In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 3, figsize=(12, 3.2))
history_df.plot(x="epoch", y="loss", marker="o", ax=axes[0], legend=False)
history_df.plot(x="epoch", y="feature_std", marker="o", ax=axes[1], legend=False)
history_df.plot(x="epoch", y="mean_pair_cosine", marker="o", ax=axes[2], legend=False)
axes[0].set_title("Latent cross-entropy")
axes[1].set_title("Mean feature std")
axes[2].set_title("Mean pair cosine")
plt.tight_layout()
figure.savefig(
    ARTIFACT_DIR / "training_diagnostics.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()


## Interpret the diagnostics carefully

- Finite, decreasing loss is necessary but not sufficient.
- Feature standard deviation near zero is a collapse warning.
- Pairwise cosine near one is a collapse warning.
- Very low target entropy across all tokens can indicate an overly sharp or degenerate teacher.
- A two-epoch smoke result only validates code flow.

In real mode, all normal GAVD sequences currently share one YouTube video. Pretraining on every normal sequence is useful for representation learning, but any later normal test drawn from that video is transductive and must be labelled that way.
